# FlowGraphFormer — 02_Baseline_Models.ipynb

Trains conventional baseline classifiers against the NumPy exports produced by
`01_Preprocessing_local.ipynb`. Run this notebook after preprocessing and before
interpreting the more complex models in `03_Big_Modeling.ipynb`.

The default uses the full exported training set. Set `TRAIN_FRACTION` in the
dataset-reduction cell to a value below `1.0` only for a stratified smoke test.
XGBoost, LightGBM, CatBoost, and TabNet are optional; their availability is
reported by the imports cell. The CatBoost and TabNet execution cells remain
disabled by default to avoid duplicating the main CatBoost experiment or
starting an unexpectedly expensive run.


In [ ]:
# ==============================================================================
# Cell 1 : Environment Setup (Colab + Local)
# ==============================================================================

print("=" * 80)
print("FlowGraphFormer - Baseline Machine Learning")
print("=" * 80)

from pathlib import Path
import os
import random
import warnings
import numpy as np

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    RUN_ENVIRONMENT = "colab"
    PROJECT_ROOT = Path("/content/drive/MyDrive/ML_Project/FlowGraphFormer")
except ModuleNotFoundError:
    RUN_ENVIRONMENT = "local"
    PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "01_Preprocessing_local.ipynb").exists():
    raise FileNotFoundError(
        f"PROJECT_ROOT does not look like this repository: {PROJECT_ROOT}. "
        "Launch Jupyter from the project directory or edit PROJECT_ROOT."
    )

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

PROCESSED_DATASET_DIR = PROJECT_ROOT / "dataset" / "processed"
FINAL_DATASET_DIR = PROCESSED_DATASET_DIR / "final"
REPORT_DIR = PROJECT_ROOT / "reports"
BASELINE_REPORT_DIR = REPORT_DIR / "baseline_ml"
FIGURE_DIR = BASELINE_REPORT_DIR / "figures"
TABLE_DIR = BASELINE_REPORT_DIR / "tables"
METRIC_DIR = BASELINE_REPORT_DIR / "metrics"
MODEL_DIR = PROJECT_ROOT / "models" / "baseline_ml"
RESULT_DIR = PROJECT_ROOT / "results" / "baseline_ml"
TEST_DIR = REPORT_DIR / "tests"

directories = [BASELINE_REPORT_DIR, FIGURE_DIR, TABLE_DIR, METRIC_DIR,
               MODEL_DIR, RESULT_DIR, TEST_DIR]
for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

print("Run Environment:", RUN_ENVIRONMENT)
print("Project Root   :", PROJECT_ROOT)
print("Environment Ready.")
print("=" * 80)


In [ ]:
# ==============================================================================
# Cell 2 : Imports
# ==============================================================================

print("=" * 80)
print("Importing Libraries")
print("=" * 80)

# ==============================================================================
# Python
# ==============================================================================

import os
import json
import time
import random
import warnings
from pathlib import Path
from datetime import datetime

# ==============================================================================
# Data Processing
# ==============================================================================

import numpy as np
import pandas as pd

# ==============================================================================
# Visualization
# ==============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 120

# ==============================================================================
# Model Persistence
# ==============================================================================

import joblib

# ==============================================================================
# Machine Learning Models
# ==============================================================================

from sklearn.ensemble import (

    RandomForestClassifier,

    ExtraTreesClassifier,

    HistGradientBoostingClassifier,

)

from sklearn.neural_network import (

    MLPClassifier,

)

# ==============================================================================
# Evaluation
# ==============================================================================

from sklearn.metrics import (

    accuracy_score,

    balanced_accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    roc_auc_score,

    matthews_corrcoef,

    cohen_kappa_score,

    confusion_matrix,

    classification_report,

    roc_curve,

    precision_recall_curve,

    auc,

    log_loss,

)

# ==============================================================================
# Timing
# ==============================================================================

from time import perf_counter

# ==============================================================================
# Progress Bar
# ==============================================================================

from tqdm.auto import tqdm

# ==============================================================================
# Warnings
# ==============================================================================

warnings.filterwarnings("ignore")

# ==============================================================================
# Optional Libraries
# ==============================================================================

HAS_XGBOOST = False
HAS_LIGHTGBM = False
HAS_CATBOOST = False
HAS_TABNET = False

try:

    from xgboost import XGBClassifier

    HAS_XGBOOST = True

except:

    pass

try:

    from lightgbm import LGBMClassifier

    HAS_LIGHTGBM = True

except:

    pass

try:

    from catboost import CatBoostClassifier

    HAS_CATBOOST = True

except:

    pass

try:

    from pytorch_tabnet.tab_model import TabNetClassifier

    HAS_TABNET = True

except:

    pass

# ==============================================================================
# Version Information
# ==============================================================================

print(f"Numpy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")

print()

print("Installed Libraries")

print("-" * 80)

print(f"XGBoost      : {HAS_XGBOOST}")
print(f"LightGBM     : {HAS_LIGHTGBM}")
print(f"CatBoost     : {HAS_CATBOOST}")
print(f"TabNet       : {HAS_TABNET}")

print()

print("Imports Completed Successfully.")

print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 3 : Load Preprocessed Dataset
# ==============================================================================

print("=" * 80)
print("Loading Preprocessed Dataset")
print("=" * 80)

# ==============================================================================
# Dataset Paths
# ==============================================================================

DATASET_DIR = PROCESSED_DATASET_DIR / "final"

# ==============================================================================
# Required Files
# ==============================================================================

required_files = {

    "X_train": DATASET_DIR / "X_train.npy",

    "X_validation": DATASET_DIR / "X_validation.npy",

    "X_test": DATASET_DIR / "X_test.npy",

    "y_train": DATASET_DIR / "y_train.npy",

    "y_validation": DATASET_DIR / "y_validation.npy",

    "y_test": DATASET_DIR / "y_test.npy",

    "metadata": DATASET_DIR / "metadata.json",

}

missing_files = []

for name, file in required_files.items():

    if not file.exists():

        missing_files.append(file)

if missing_files:

    raise FileNotFoundError(

        "Missing required files:\n"

        + "\n".join(str(f) for f in missing_files)

    )

# ==============================================================================
# Load Dataset
# ==============================================================================

print("Loading NumPy arrays...\n")

X_train = np.load(

    required_files["X_train"]

)

X_validation = np.load(

    required_files["X_validation"]

)

X_test = np.load(

    required_files["X_test"]

)

y_train = np.load(

    required_files["y_train"]

)

y_validation = np.load(

    required_files["y_validation"]

)

y_test = np.load(

    required_files["y_test"]

)

# ==============================================================================
# Load Metadata
# ==============================================================================

with open(

    required_files["metadata"],

    "r",

) as f:

    metadata = json.load(f)

# ==============================================================================
# Dataset Summary
# ==============================================================================

summary = pd.DataFrame({

    "Dataset":[

        "Training",

        "Validation",

        "Testing",

    ],

    "Samples":[

        len(X_train),

        len(X_validation),

        len(X_test),

    ],

    "Features":[

        X_train.shape[1],

        X_validation.shape[1],

        X_test.shape[1],

    ],

})

display(summary)

# ==============================================================================
# Print Metadata
# ==============================================================================

print()

print("Metadata")

print("-" * 80)

for key, value in metadata.items():

    if isinstance(value, list):

        print(f"{key:<20}: {len(value)} items")

    else:

        print(f"{key:<20}: {value}")

# ==============================================================================
# Dataset Validation
# ==============================================================================

assert X_train.shape[1] == X_validation.shape[1]
assert X_train.shape[1] == X_test.shape[1]

assert len(X_train) == len(y_train)
assert len(X_validation) == len(y_validation)
assert len(X_test) == len(y_test)

assert np.isnan(X_train).sum() == 0
assert np.isnan(X_validation).sum() == 0
assert np.isnan(X_test).sum() == 0

# ==============================================================================
# Number of Classes
# ==============================================================================

num_classes = len(np.unique(y_train))

num_features = X_train.shape[1]

print()

print(f"Training Samples   : {len(X_train):,}")
print(f"Validation Samples : {len(X_validation):,}")
print(f"Testing Samples    : {len(X_test):,}")

print()

print(f"Features           : {num_features}")
print(f"Classes            : {num_classes}")

print()

print("Dataset Loaded Successfully.")

print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 4 : Optional Stratified Smoke-Test Reduction
# ==============================================================================

from sklearn.model_selection import train_test_split

# 1.0 keeps the complete training set. Use, for example, 0.02 only for a
# quick smoke test; validation and test sets always remain untouched.
TRAIN_FRACTION = 1.0

if not 0 < TRAIN_FRACTION <= 1:
    raise ValueError("TRAIN_FRACTION must be in the interval (0, 1].")

if TRAIN_FRACTION < 1.0:
    X_train, _, y_train, _ = train_test_split(
        X_train,
        y_train,
        train_size=TRAIN_FRACTION,
        stratify=y_train,
        random_state=SEED,
    )
    print(f"Smoke-test training samples: {len(X_train):,} ({TRAIN_FRACTION:.1%})")
else:
    print(f"Full training set retained: {len(X_train):,} samples")


In [ ]:
# ==============================================================================
# Cell 4 : Configuration
# ==============================================================================

print("=" * 80)
print("Model Configuration")
print("=" * 80)

CONFIG = {

    # ==========================================================================
    # General
    # ==========================================================================

    "random_state": SEED,

    "n_jobs": -1,

    "verbosity": 0,

    "save_models": True,

    "save_predictions": True,

    "save_metrics": True,

    "save_figures": True,

    # ==========================================================================
    # Evaluation
    # ==========================================================================

    "average": "weighted",

    "multi_class": "ovr",

    "probability_threshold": 0.50,

    # ==========================================================================
    # Random Forest
    # ==========================================================================

    "RandomForest": {

        "n_estimators": 300,

        "max_depth": None,

        "min_samples_split": 2,

        "min_samples_leaf": 1,

        "max_features": "sqrt",

        "bootstrap": True,

        "random_state": SEED,

        "n_jobs": -1,

    },

    # ==========================================================================
    # Extra Trees
    # ==========================================================================

    "ExtraTrees": {

        "n_estimators": 300,

        "max_depth": None,

        "max_features": "sqrt",

        "random_state": SEED,

        "n_jobs": -1,

    },

    # ==========================================================================
    # HistGradientBoosting
    # ==========================================================================

    "HistGradientBoosting": {

        "learning_rate": 0.1,

        "max_iter": 300,

        "max_depth": None,

        "random_state": SEED,

    },

    # ==========================================================================
    # MLP
    # ==========================================================================

    "MLP": {

        "hidden_layer_sizes": (256, 128),

        "activation": "relu",

        "solver": "adam",

        "learning_rate": "adaptive",

        "max_iter": 100,

        "random_state": SEED,

    },

    # ==========================================================================
    # XGBoost
    # ==========================================================================

    "XGBoost": {

        "n_estimators": 300,

        "max_depth": 8,

        "learning_rate": 0.05,

        "subsample": 0.8,

        "colsample_bytree": 0.8,

        "tree_method": "hist",

        "random_state": SEED,

        "n_jobs": -1,

    },

    # ==========================================================================
    # LightGBM
    # ==========================================================================

    "LightGBM": {

        "n_estimators": 300,

        "learning_rate": 0.05,

        "num_leaves": 63,

        "subsample": 0.8,

        "colsample_bytree": 0.8,

        "random_state": SEED,

        "n_jobs": -1,

    },

    # ==========================================================================
    # CatBoost
    # ==========================================================================

    "CatBoost": {

        "iterations": 300,

        "depth": 8,

        "learning_rate": 0.05,

        "loss_function": "MultiClass",

        "verbose": False,

        "random_state": SEED,

    },

    # ==========================================================================
    # TabNet
    # ==========================================================================

    "TabNet": {

        "n_d": 32,

        "n_a": 32,

        "n_steps": 5,

        "gamma": 1.5,

        "lambda_sparse": 1e-4,

        "seed": SEED,

    }

}

# ==============================================================================
# Models To Train
# ==============================================================================

MODEL_LIST = [

    "RandomForest",

    "XGBoost",

    "LightGBM",

    "CatBoost",

    "ExtraTrees",

    "HistGradientBoosting",

    "MLP",

    "TabNet",

]

# ==============================================================================
# Summary
# ==============================================================================

summary = pd.DataFrame({

    "Model": MODEL_LIST,

    "Enabled": [True] * len(MODEL_LIST)

})

display(summary)

print()

print(f"Total Models : {len(MODEL_LIST)}")

print()

print("Configuration Loaded Successfully.")

print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 5 : Evaluation Utilities
# ==============================================================================

print("=" * 80)
print("Creating Evaluation Utilities")
print("=" * 80)

# ==============================================================================
# Evaluate Classification Model
# ==============================================================================

def evaluate_model(

    model,

    X_test,

    y_test,

    train_time=0.0,

    prediction_time=0.0,

):

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):

        y_prob = model.predict_proba(X_test)

    else:

        y_prob = None

    metrics = {}

    metrics["Accuracy"] = accuracy_score(

        y_test,

        y_pred,

    )

    metrics["Balanced Accuracy"] = balanced_accuracy_score(

        y_test,

        y_pred,

    )

    metrics["Precision"] = precision_score(

        y_test,

        y_pred,

        average=CONFIG["average"],

        zero_division=0,

    )

    metrics["Recall"] = recall_score(

        y_test,

        y_pred,

        average=CONFIG["average"],

        zero_division=0,

    )

    metrics["F1 Score"] = f1_score(

        y_test,

        y_pred,

        average=CONFIG["average"],

        zero_division=0,

    )

    metrics["MCC"] = matthews_corrcoef(

        y_test,

        y_pred,

    )

    metrics["Cohen Kappa"] = cohen_kappa_score(

        y_test,

        y_pred,

    )

    metrics["Training Time (s)"] = train_time

    metrics["Prediction Time (s)"] = prediction_time

    metrics["Samples / Second"] = (

        len(X_test) / prediction_time

        if prediction_time > 0 else 0

    )

    if y_prob is not None:

        try:

            metrics["ROC AUC"] = roc_auc_score(

                y_test,

                y_prob,

                multi_class=CONFIG["multi_class"],

                average=CONFIG["average"],

            )

        except:

            metrics["ROC AUC"] = np.nan

        try:

            metrics["Log Loss"] = log_loss(

                y_test,

                y_prob,

            )

        except:

            metrics["Log Loss"] = np.nan

    else:

        metrics["ROC AUC"] = np.nan

        metrics["Log Loss"] = np.nan

    return {

        "metrics": metrics,

        "predictions": y_pred,

        "probabilities": y_prob,

        "confusion_matrix": confusion_matrix(

            y_test,

            y_pred,

        ),

        "classification_report": classification_report(

            y_test,

            y_pred,

            output_dict=True,

            zero_division=0,

        ),

    }

# ==============================================================================
# Save Metrics
# ==============================================================================

def save_metrics(

    results,

    output_directory,

):

    output_directory.mkdir(

        parents=True,

        exist_ok=True,

    )

    metrics_df = pd.DataFrame(

        [results["metrics"]]

    )

    metrics_df.to_csv(

        output_directory / "metrics.csv",

        index=False,

    )

    with open(

        output_directory / "metrics.json",

        "w",

    ) as f:

        json.dump(

            results["metrics"],

            f,

            indent=4,

        )

    report_df = pd.DataFrame(

        results["classification_report"]

    ).transpose()

    report_df.to_csv(

        output_directory / "classification_report.csv",

    )

    cm = pd.DataFrame(

        results["confusion_matrix"]

    )

    cm.to_csv(

        output_directory / "confusion_matrix.csv",

        index=False,

    )

# ==============================================================================
# Plot Confusion Matrix
# ==============================================================================

def plot_confusion_matrix(

    cm,

    title,

    save_path,

):

    plt.figure(figsize=(8, 6))

    sns.heatmap(

        cm,

        annot=True,

        fmt="d",

        cmap="Blues",

    )

    plt.title(title)

    plt.xlabel("Predicted")

    plt.ylabel("Actual")

    plt.tight_layout()

    plt.savefig(

        save_path,

        dpi=300,

    )

    plt.close()

# ==============================================================================
# Save Trained Model
# ==============================================================================

def save_model(

    model,

    save_path,

):

    joblib.dump(

        model,

        save_path,

    )

# ==============================================================================
# Summary
# ==============================================================================

print("Functions Created Successfully")

print()

print("✓ evaluate_model()")

print("✓ save_metrics()")

print("✓ plot_confusion_matrix()")

print("✓ save_model()")

print()

print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 6 : Model Factory
# ==============================================================================

print("=" * 80)
print("Creating Model Factory")
print("=" * 80)

# ==============================================================================
# Create Model
# ==============================================================================

def create_model(model_name):

    if model_name == "RandomForest":

        return RandomForestClassifier(

            **CONFIG["RandomForest"]

        )

    elif model_name == "ExtraTrees":

        return ExtraTreesClassifier(

            **CONFIG["ExtraTrees"]

        )

    elif model_name == "HistGradientBoosting":

        return HistGradientBoostingClassifier(

            **CONFIG["HistGradientBoosting"]

        )

    elif model_name == "MLP":

        return MLPClassifier(

            **CONFIG["MLP"]

        )

    elif model_name == "XGBoost":

        if not HAS_XGBOOST:

            raise ImportError(

                "XGBoost is not installed."

            )

        return XGBClassifier(

            **CONFIG["XGBoost"]

        )

    elif model_name == "LightGBM":

        if not HAS_LIGHTGBM:

            raise ImportError(

                "LightGBM is not installed."

            )

        return LGBMClassifier(

            **CONFIG["LightGBM"]

        )

    elif model_name == "CatBoost":

        if not HAS_CATBOOST:

            raise ImportError(

                "CatBoost is not installed."

            )

        return CatBoostClassifier(

            **CONFIG["CatBoost"]

        )

    elif model_name == "TabNet":

        if not HAS_TABNET:

            raise ImportError(

                "TabNet is not installed."

            )

        return TabNetClassifier(

            **CONFIG["TabNet"]

        )

    else:

        raise ValueError(

            f"Unknown model : {model_name}"

        )

# ==============================================================================
# Available Models
# ==============================================================================

available_models = []

for model_name in MODEL_LIST:

    try:

        _ = create_model(model_name)

        available_models.append({

            "Model": model_name,

            "Available": True,

        })

    except Exception:

        available_models.append({

            "Model": model_name,

            "Available": False,

        })

available_models = pd.DataFrame(

    available_models

)

display(

    available_models

)

# ==============================================================================
# Test Factory
# ==============================================================================

print()

print("Factory Test")

print("-" * 80)

for model_name in MODEL_LIST:

    try:

        model = create_model(

            model_name

        )

        print(f"{model_name:<25} PASS")

    except Exception as e:

        print(f"{model_name:<25} FAIL")

        print(f"    {e}")

print()

print("=" * 80)

print("Model Factory Ready")

print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 7 : Universal Training Pipeline
# ==============================================================================

print("=" * 80)
print("Creating Universal Training Pipeline")
print("=" * 80)

# ==============================================================================
# Global Results
# ==============================================================================

ALL_RESULTS = {}

# ==============================================================================
# Universal Trainer
# ==============================================================================

def train_model(model_name):

    print("\n" + "=" * 80)
    print(f"Training : {model_name}")
    print("=" * 80)

    model = create_model(model_name)

    model_output_dir = RESULT_DIR / model_name
    model_output_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------------------------
    # Train
    # --------------------------------------------------------------------------

    train_start = perf_counter()

    if model_name == "TabNet":

        model.fit(

            X_train,

            y_train,

            eval_set=[(X_validation, y_validation)],

            max_epochs=100,

            patience=10,

            batch_size=4096,

            virtual_batch_size=512,

        )

    else:

        model.fit(

            X_train,

            y_train,

        )

    train_time = perf_counter() - train_start

    # --------------------------------------------------------------------------
    # Save Model (Immediate Save after Training)
    # --------------------------------------------------------------------------

    if CONFIG["save_models"]:

        save_model(

            model,

            MODEL_DIR / f"{model_name}.pkl",

        )
        print(f"✓ Model {model_name} saved to {MODEL_DIR}")

    # --------------------------------------------------------------------------
    # Prediction
    # --------------------------------------------------------------------------

    prediction_start = perf_counter()

    results = evaluate_model(

        model,

        X_test,

        y_test,

        train_time=train_time,

        prediction_time=0,

    )

    prediction_time = perf_counter() - prediction_start

    results["metrics"]["Prediction Time (s)"] = prediction_time

    results["metrics"]["Samples / Second"] = (

        len(X_test) / prediction_time

    )

    # --------------------------------------------------------------------------
    # Save Metrics
    # --------------------------------------------------------------------------

    save_metrics(

        results,

        model_output_dir,

    )

    # --------------------------------------------------------------------------
    # Confusion Matrix
    # --------------------------------------------------------------------------

    plot_confusion_matrix(

        results["confusion_matrix"],

        title=model_name,

        save_path=FIGURE_DIR / f"{model_name}_confusion_matrix.png",

    )

    # --------------------------------------------------------------------------
    # Feature Importance
    # --------------------------------------------------------------------------

    if hasattr(model, "feature_importances_"):

        importance = pd.DataFrame({

            "Feature": range(len(model.feature_importances_)),

            "Importance": model.feature_importances_,

        })

        importance = importance.sort_values(

            "Importance",

            ascending=False,

        )

        importance.to_csv(

            model_output_dir / "feature_importance.csv",

            index=False,

        )

        plt.figure(figsize=(10,8))

        sns.barplot(

            data=importance.head(20),

            x="Importance",

            y="Feature",

        )

        plt.title(

            f"{model_name} Feature Importance",

        )

        plt.tight_layout()

        plt.savefig(

            FIGURE_DIR / f"{model_name}_feature_importance.png",

            dpi=300,

        )

        plt.close()

    # --------------------------------------------------------------------------
    # Store Result
    # --------------------------------------------------------------------------

    ALL_RESULTS[model_name] = results

    print()

    print("Accuracy :", f"{results['metrics']['Accuracy']:.4f}")

    print("F1 Score :", f"{results['metrics']['F1 Score']:.4f}")

    print("ROC AUC  :", f"{results['metrics']['ROC AUC']:.4f}")

    print()

    print(f"Training Time   : {train_time:.2f} sec")

    print(f"Prediction Time : {prediction_time:.2f} sec")

    print()

    print(f"{model_name} Completed.")

    return results

# ==============================================================================
# Summary
# ==============================================================================

print("Universal Trainer Ready")

print()

print("Usage Example:")

print()

print('results = train_model("RandomForest")')

print()

print("=" * 80)


In [ ]:
# ==============================================================================
# Cell 8 : Random Forest
# ==============================================================================

print("=" * 80)
print("Random Forest")
print("=" * 80)

rf_results = train_model(

    "RandomForest"

)

print()

print("Random Forest Finished Successfully.")

print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 9 : XGBoost
# ==============================================================================

print("=" * 80)
print("XGBoost")
print("=" * 80)

xgb_results = train_model(

    "XGBoost"

)

print()

print("XGBoost Finished Successfully.")

print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 10 : LightGBM
# ==============================================================================

print("=" * 80)
print("LightGBM")
print("=" * 80)

lgbm_results = train_model(

    "LightGBM"

)

print()

print("LightGBM Finished Successfully.")

print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 11 : CatBoost
# ==============================================================================

# print("=" * 80)
# print("CatBoost")
# print("=" * 80)

# catboost_results = train_model(

#     "CatBoost"

# )

# print()

# print("CatBoost Finished Successfully.")

# print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 12 : Extra Trees
# ==============================================================================

print("=" * 80)
print("Extra Trees")
print("=" * 80)

extra_trees_results = train_model(

    "ExtraTrees"

)

print()

print("Extra Trees Finished Successfully.")

print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 13 : HistGradientBoosting
# ==============================================================================

print("=" * 80)
print("HistGradientBoosting")
print("=" * 80)

histgb_results = train_model(

    "HistGradientBoosting"

)

print()

print("HistGradientBoosting Finished Successfully.")

print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 14 : MLP
# ==============================================================================

print("=" * 80)
print("MLP")
print("=" * 80)

mlp_results = train_model(

    "MLP"

)

print()

print("MLP Finished Successfully.")

print("=" * 80)

In [ ]:
# # ==============================================================================
# # Cell 15 : TabNet
# # ==============================================================================

# print("=" * 80)
# print("TabNet")
# print("=" * 80)

# tabnet_results = train_model(

#     "TabNet"

# )

# print()

# print("TabNet Finished Successfully.")

# print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 16 : Universal Model Comparison
# ==============================================================================

print("=" * 80)
print("Comparing Baseline Models")
print("=" * 80)

comparison = []

# ==============================================================================
# Collect Results
# ==============================================================================

for model_name, result in ALL_RESULTS.items():

    row = {

        "Model": model_name,

    }

    row.update(

        result["metrics"]

    )

    comparison.append(

        row

    )

comparison_df = pd.DataFrame(

    comparison

)

# ==============================================================================
# Ranking
# ==============================================================================

comparison_df["Accuracy Rank"] = (

    comparison_df["Accuracy"]

    .rank(

        ascending=False,

        method="min",

    )

)

comparison_df["F1 Rank"] = (

    comparison_df["F1 Score"]

    .rank(

        ascending=False,

        method="min",

    )

)

comparison_df["ROC Rank"] = (

    comparison_df["ROC AUC"]

    .rank(

        ascending=False,

        method="min",

    )

)

comparison_df["MCC Rank"] = (

    comparison_df["MCC"]

    .rank(

        ascending=False,

        method="min",

    )

)

comparison_df["Overall Rank"] = (

    comparison_df[

        [

            "Accuracy Rank",

            "F1 Rank",

            "ROC Rank",

            "MCC Rank",

        ]

    ]

    .mean(axis=1)

)

comparison_df = comparison_df.sort_values(

    "Overall Rank"

)

display(

    comparison_df

)

# ==============================================================================
# Save Results
# ==============================================================================

comparison_df.to_csv(

    TABLE_DIR / "comparison_metrics.csv",

    index=False,

)

comparison_df.to_excel(

    TABLE_DIR / "comparison_metrics.xlsx",

    index=False,

)

comparison_df.to_json(

    TABLE_DIR / "comparison_metrics.json",

    orient="records",

    indent=4,

)

# ==============================================================================
# Best Models
# ==============================================================================

best_models = {

    "Accuracy":

        comparison_df.loc[

            comparison_df["Accuracy"].idxmax()

        ]["Model"],

    "F1":

        comparison_df.loc[

            comparison_df["F1 Score"].idxmax()

        ]["Model"],

    "ROC":

        comparison_df.loc[

            comparison_df["ROC AUC"].idxmax()

        ]["Model"],

    "MCC":

        comparison_df.loc[

            comparison_df["MCC"].idxmax()

        ]["Model"],

    "Fastest Training":

        comparison_df.loc[

            comparison_df["Training Time (s)"].idxmin()

        ]["Model"],

    "Fastest Prediction":

        comparison_df.loc[

            comparison_df["Prediction Time (s)"].idxmin()

        ]["Model"],

}

with open(

    TABLE_DIR / "best_models.json",

    "w",

) as f:

    json.dump(

        best_models,

        f,

        indent=4,

    )

print()

print("Best Models")

print("-" * 80)

for metric, model in best_models.items():

    print(f"{metric:<25}: {model}")

# ==============================================================================
# Summary Statistics
# ==============================================================================

summary = comparison_df.describe()

summary.to_csv(

    TABLE_DIR / "comparison_summary.csv"

)

print()

print("Comparison Completed Successfully.")

print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 17 : Performance Visualization
# ==============================================================================

print("=" * 80)
print("Generating Performance Figures")
print("=" * 80)

metrics_to_plot = [

    "Accuracy",

    "Precision",

    "Recall",

    "F1 Score",

    "ROC AUC",

    "MCC",

    "Training Time (s)",

    "Prediction Time (s)",

]

comparison = pd.read_csv(

    TABLE_DIR / "comparison_metrics.csv"

)

# ==============================================================================
# Individual Bar Charts
# ==============================================================================

for metric in metrics_to_plot:

    plt.figure(figsize=(10,6))

    chart = comparison.sort_values(

        metric,

        ascending=False

    )

    sns.barplot(

        data=chart,

        x="Model",

        y=metric,

    )

    plt.xticks(

        rotation=30,

        ha="right",

    )

    plt.title(

        metric,

        fontsize=16,

        fontweight="bold",

    )

    plt.tight_layout()

    plt.savefig(

        FIGURE_DIR / f"{metric.replace(' ','_')}.png",

        dpi=300,

    )

    plt.close()

# ==============================================================================
# Heatmap
# ==============================================================================

heatmap_metrics = [

    "Accuracy",

    "Precision",

    "Recall",

    "F1 Score",

    "ROC AUC",

    "MCC",

]

heatmap_df = comparison.set_index(

    "Model"

)[heatmap_metrics]

plt.figure(figsize=(10,6))

sns.heatmap(

    heatmap_df,

    annot=True,

    cmap="YlGnBu",

    fmt=".4f",

)

plt.title(

    "Model Comparison Heatmap",

    fontsize=16,

)

plt.tight_layout()

plt.savefig(

    FIGURE_DIR / "comparison_heatmap.png",

    dpi=300,

)

plt.close()

# ==============================================================================
# Radar Chart
# ==============================================================================

from math import pi

radar_metrics = [

    "Accuracy",

    "Precision",

    "Recall",

    "F1 Score",

    "ROC AUC",

    "MCC",

]

angles = np.linspace(

    0,

    2*np.pi,

    len(radar_metrics),

    endpoint=False,

).tolist()

angles += angles[:1]

plt.figure(figsize=(8,8))

ax = plt.subplot(

    polar=True

)

for _, row in comparison.iterrows():

    values = [

        row[m]

        for m in radar_metrics

    ]

    values += values[:1]

    ax.plot(

        angles,

        values,

        linewidth=2,

        label=row["Model"],

    )

ax.set_xticks(

    angles[:-1]

)

ax.set_xticklabels(

    radar_metrics

)

plt.legend(

    bbox_to_anchor=(1.25,1.0)

)

plt.title(

    "Radar Comparison",

    fontsize=16,

)

plt.tight_layout()

plt.savefig(

    FIGURE_DIR / "radar_chart.png",

    dpi=300,

)

plt.close()

print()

print("Figures Generated Successfully.")

print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 18 : Save Best Model
# ==============================================================================

print("=" * 80)
print("Saving Best Model")
print("=" * 80)

# ==============================================================================
# Load Comparison Table
# ==============================================================================

comparison = pd.read_csv(

    TABLE_DIR / "comparison_metrics.csv"

)

# ==============================================================================
# Best Model (Highest F1)
# ==============================================================================

best_row = comparison.sort_values(

    "F1 Score",

    ascending=False,

).iloc[0]

best_model_name = best_row["Model"]

print(f"Best Model : {best_model_name}")

# ==============================================================================
# Directories
# ==============================================================================

BEST_MODEL_DIR = MODEL_DIR / "best"

BEST_MODEL_DIR.mkdir(

    parents=True,

    exist_ok=True,

)

# ==============================================================================
# Copy Model
# ==============================================================================

import shutil

source_model = MODEL_DIR / f"{best_model_name}.pkl"

destination_model = BEST_MODEL_DIR / f"{best_model_name}.pkl"

shutil.copy2(

    source_model,

    destination_model,

)

# ==============================================================================
# Save Metadata
# ==============================================================================

metadata = {

    "model": best_model_name,

    "accuracy": float(best_row["Accuracy"]),

    "precision": float(best_row["Precision"]),

    "recall": float(best_row["Recall"]),

    "f1_score": float(best_row["F1 Score"]),

    "roc_auc": float(best_row["ROC AUC"]),

    "mcc": float(best_row["MCC"]),

    "training_time": float(best_row["Training Time (s)"]),

    "prediction_time": float(best_row["Prediction Time (s)"]),

    "created": datetime.now().strftime(

        "%Y-%m-%d %H:%M:%S"

    )

}

with open(

    BEST_MODEL_DIR / "metadata.json",

    "w",

) as f:

    json.dump(

        metadata,

        f,

        indent=4,

    )

# ==============================================================================
# Save Best Model Metrics
# ==============================================================================

best_row.to_frame().T.to_csv(

    BEST_MODEL_DIR / "metrics.csv",

    index=False,

)

print()

print("Best Model Saved Successfully.")

print("=" * 80)